In [22]:
import pandas as pd

# ==============================================================================
# DATA
# ==============================================================================
df_penyakitmenular = pd.read_csv('jumlah_penyakit_menular.csv')

df_penyakitmenular

,id,id_index,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,jumlah,satuan,tahun
0,1,11,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,HIV,15.0,KASUS,2018
1,1,21,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,DEMAM BERDARAH,268.0,KASUS,2018
2,1,31,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,KUSTA,9.0,KASUS,2018
3,1,41,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,COVID 19,0.0,KASUS,2018
4,2,52,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018,COVID 19,0.0,KASUS,2018
...,...,...,...,...,...,...,...,...,...,...,...
1059,265,1060265,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,KUSTA,90.0,KASUS,2024
1060,266,1061266,35,JAWA TIMUR,3579,KOTA BATU,2024,HIV,63.0,KASUS,2024
1061,266,1062266,35,JAWA TIMUR,3579,KOTA BATU,2024,DEMAM BERDARAH,444.0,KASUS,2024
1062,266,1063266,35,JAWA TIMUR,3579,KOTA BATU,2024,KUSTA,2.0,KASUS,2024


DATA CLEANING

In [23]:
#===============================================================================
# DATA CLEANING
#===============================================================================

# -------------------------
# 1. INFO DATA
# -------------------------
print("\n 1. Info Data:")
df_penyakitmenular.info()

print("\n Info Data Setelah Konversi ke String (selain jumlah dan tahun)")
kolom_string = [
    'id',
    'id_index',
    'kode_provinsi',
    'kode_kabupaten_kota',
    'periode_update'
]

df_penyakitmenular[kolom_string] = df_penyakitmenular[kolom_string].astype(str)
df_penyakitmenular.info()

# -------------------------
# 2. STANDARDISASI KAB/KOT
# -------------------------
df_penyakitmenular['nama_kabupaten_kota'] = (
    df_penyakitmenular['nama_kabupaten_kota']
    .str.upper()
    .str.strip()
)

daftar_kabkot = [
    'KABUPATEN PACITAN',
    'KABUPATEN PONOROGO',
    'KABUPATEN TRENGGALEK',
    'KABUPATEN TULUNGAGUNG',
    'KABUPATEN BLITAR',
    'KABUPATEN KEDIRI',
    'KABUPATEN MALANG',
    'KABUPATEN LUMAJANG',
    'KABUPATEN JEMBER',
    'KABUPATEN BANYUWANGI',
    'KABUPATEN BONDOWOSO',
    'KABUPATEN SITUBONDO',
    'KABUPATEN PROBOLINGGO',
    'KABUPATEN PASURUAN',
    'KABUPATEN SIDOARJO',
    'KABUPATEN MOJOKERTO',
    'KABUPATEN JOMBANG',
    'KABUPATEN NGANJUK',
    'KABUPATEN MADIUN',
    'KABUPATEN MAGETAN',
    'KABUPATEN NGAWI',
    'KABUPATEN BOJONEGORO',
    'KABUPATEN TUBAN',
    'KABUPATEN LAMONGAN',
    'KABUPATEN GRESIK',
    'KABUPATEN BANGKALAN',
    'KABUPATEN SAMPANG',
    'KABUPATEN PAMEKASAN',
    'KABUPATEN SUMENEP',
    'KOTA KEDIRI',
    'KOTA BLITAR',
    'KOTA MALANG',
    'KOTA PROBOLINGGO',
    'KOTA PASURUAN',
    'KOTA MOJOKERTO',
    'KOTA MADIUN',
    'KOTA SURABAYA',
    'KOTA BATU'
]

# -------------------------
# 3. CEK KELENGKAPAN KAB/KOT
# -------------------------
tidak_ada = [
    nama for nama in daftar_kabkot
    if nama not in df_penyakitmenular['nama_kabupaten_kota'].values
]

print("\n2.Cek Kelengkapan Kabupaten/Kota")

if len(tidak_ada) > 0:
    print("Nama Kabupaten/kota yang tidak ada di dataset:")
    for nama in tidak_ada:
        print(nama)
else:
    print("Semua nama kabupaten/kota tersedia di dataset")

# -------------------------
# 4. CEK DUPLIKAT
# -------------------------
jumlah_duplikat = df_penyakitmenular.duplicated().sum()

print("\n3.Cek Data Duplikat")

if jumlah_duplikat > 0:
    print("Jumlah data duplikat:", jumlah_duplikat)
    df_penyakitmenular = df_penyakitmenular.drop_duplicates()
    print("Berhasil dihapus")
else:
    print("Tidak ada data duplikat")


# -------------------------
# 5. CEK MISSING VALUE
# -------------------------
missing_value = df_penyakitmenular.isnull().sum()

print("\n4.Cek Missing Value")
for kolom in df_penyakitmenular.columns:
    indeks_nan = df_penyakitmenular[df_penyakitmenular[kolom].isna()].index.tolist()

    if len(indeks_nan) > 0:
        print(f"{kolom}: {indeks_nan}")

if missing_value.sum() > 0:
    print("Jumlah missing value:", missing_value.sum())

    kolom_kategori = 'kategori'
    kolom_numerik = df_penyakitmenular.select_dtypes(include='number').columns

    for kolom in kolom_numerik:
        if df_penyakitmenular[kolom].isna().sum() > 0:
            df_penyakitmenular[kolom] = df_penyakitmenular.groupby(kolom_kategori)[kolom].transform(
                lambda x: x.fillna(x.median())
            )

    print("Berhasil ditangani sesuai kategori")

else:
    print("Tidak ada missing value")



# -------------------------
# 6. CEK OUTLIER (IQR)
# -------------------------
hasil_outlier = []
kolom_numerik = df_penyakitmenular.select_dtypes(include='number').columns

print("\n5.Cek Outlier (IQR)")

for kolom in kolom_numerik:

    Q1 = df_penyakitmenular[kolom].quantile(0.25)
    Q3 = df_penyakitmenular[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    jumlah_outlier = df_penyakitmenular[
        (df_penyakitmenular[kolom] < batas_bawah) |
        (df_penyakitmenular[kolom] > batas_atas)
    ]

    if len(jumlah_outlier) > 0:
        keterangan = "Ada outlier"
        print(f"\nVariabel: {kolom}")
        print(f'Jumlah Outlier: {len(jumlah_outlier)}')
        display(jumlah_outlier[['nama_kabupaten_kota', 'tahun', kolom]])
    else:
        print(f"\nVariabel: {kolom}")
        print("Tidak ada outlier")


 1. Info Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1064 entries, 0 to 1063
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   1064 non-null   int64  
 1   id_index             1064 non-null   int64  
 2   kode_provinsi        1064 non-null   int64  
 3   nama_provinsi        1064 non-null   object 
 4   kode_kabupaten_kota  1064 non-null   int64  
 5   nama_kabupaten_kota  1064 non-null   object 
 6   periode_update       1064 non-null   int64  
 7   kategori             1064 non-null   object 
 8   jumlah               1063 non-null   float64
 9   satuan               1064 non-null   object 
 10  tahun                1064 non-null   int64  
dtypes: float64(1), int64(6), object(4)
memory usage: 91.6+ KB

 Info Data Setelah Konversi ke String (selain jumlah dan tahun)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1064 entries, 0 to 1063
Data columns (total 11 co

,nama_kabupaten_kota,tahun,jumlah
82,KABUPATEN NGAWI,2018,827.0
158,KABUPATEN PONOROGO,2019,1721.0
165,KABUPATEN TULUNGAGUNG,2019,899.0
175,KABUPATEN KEDIRI,2019,1398.0
176,KABUPATEN MALANG,2019,1570.0
...,...,...,...
1006,KABUPATEN LAMONGAN,2024,888.0
1018,KABUPATEN SAMPANG,2024,830.0
1020,KABUPATEN PAMEKASAN,2024,972.0
1027,KABUPATEN SUMENEP,2024,1532.0



Variabel: tahun
Tidak ada outlier


In [24]:
df_penyakitmenular

,id,id_index,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,jumlah,satuan,tahun
0,1,11,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,HIV,15.0,KASUS,2018
1,1,21,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,DEMAM BERDARAH,268.0,KASUS,2018
2,1,31,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,KUSTA,9.0,KASUS,2018
3,1,41,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,COVID 19,0.0,KASUS,2018
4,2,52,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018,COVID 19,0.0,KASUS,2018
...,...,...,...,...,...,...,...,...,...,...,...
1059,265,1060265,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,KUSTA,90.0,KASUS,2024
1060,266,1061266,35,JAWA TIMUR,3579,KOTA BATU,2024,HIV,63.0,KASUS,2024
1061,266,1062266,35,JAWA TIMUR,3579,KOTA BATU,2024,DEMAM BERDARAH,444.0,KASUS,2024
1062,266,1063266,35,JAWA TIMUR,3579,KOTA BATU,2024,KUSTA,2.0,KASUS,2024


TRANSFORMASI DATA

In [25]:
# ==============================================================================
# AGGREGASI JUMLAH PENYAKIT MENULAR PER KABUPATEN/KOTA PER TAHUN
# ==============================================================================

# -------------------------
# JUMLAH PENYAKIT MENULAR PER KABUPATEN/KOTA PER TAHUN
# -------------------------
df_penyakitmenular_tahun_kab = (
    df_penyakitmenular.groupby([
        'kode_kabupaten_kota',
        'nama_kabupaten_kota',
        'tahun'
    ])['jumlah']
    .sum()
    .reset_index()
)
df_penyakitmenular_tahun_kab = df_penyakitmenular_tahun_kab.rename(columns={
    'jumlah': 'JUMLAH_PENYAKIT_MENULAR'
})

df_penyakitmenular_tahun_kab = df_penyakitmenular_tahun_kab.sort_values(
    ['tahun', 'kode_kabupaten_kota'],
    ascending=[True, True]
).reset_index(drop=True)

df_penyakitmenular_tahun_kab['JUMLAH_PENYAKIT_MENULAR'] = (
    df_penyakitmenular_tahun_kab['JUMLAH_PENYAKIT_MENULAR']
    .round()
    .astype('Int64')
)

df_penyakitmenular_tahun_kab



,kode_kabupaten_kota,nama_kabupaten_kota,tahun,JUMLAH_PENYAKIT_MENULAR
0,3501,KABUPATEN PACITAN,2018,292
1,3502,KABUPATEN PONOROGO,2018,479
2,3503,KABUPATEN TRENGGALEK,2018,285
3,3504,KABUPATEN TULUNGAGUNG,2018,507
4,3505,KABUPATEN BLITAR,2018,392
...,...,...,...,...
261,3575,KOTA PASURUAN,2024,287
262,3576,KOTA MOJOKERTO,2024,129
263,3577,KOTA MADIUN,2024,756
264,3578,KOTA SURABAYA,2024,2299


SIMPAN DATA

In [26]:
#===============================================================================
# SIMPAN DATA
#===============================================================================
df_penyakitmenular_tahun_kab.to_csv('data_jumlah_penyakitmenular.csv', index=False)

In [27]:
print(df_penyakitmenular_tahun_kab.columns.tolist())

['kode_kabupaten_kota', 'nama_kabupaten_kota', 'tahun', 'JUMLAH_PENYAKIT_MENULAR']
